In [0]:
create or replace view workspace.dimension.dim_date as 
-- ===================================================================
-- Create logic for date dimension
-- ===================================================================
SELECT
	  cast(date_format(DATE_ID, 'yyyyMMdd') AS INT) AS DATE_DIM_ID
	, DATE_ID
	-- Calendar parts
	, day(DATE_ID)                               AS DAY_OF_MONTH
	, month(DATE_ID)                             AS MONTH
	, year(DATE_ID)                              AS YEAR
	-- Month names
	, date_format(DATE_ID, 'MMM')                AS MONTH_NAME_SHORT
	, date_format(DATE_ID, 'MMMM')               AS MONTH_NAME_FULL
	-- ISO weekday / week / week-year
	, extract(DAYOFWEEK_ISO FROM DATE_ID)        AS DAY_OF_WEEK_NUM
	, upper(date_format(DATE_ID, 'E'))           AS DAY_OF_WEEK_DESC
	, weekofyear(DATE_ID)                        AS ISO_WEEK
	, concat(weekofyear(DATE_ID), '-', extract(DAYOFWEEK_ISO FROM DATE_ID)) AS SAME_DAY_KEY
	-- Week boundaries
	, cast(date_trunc('WEEK', DATE_ID) AS DATE)  AS START_OF_WEEK
	, date_add(cast(date_trunc('WEEK', DATE_ID) AS DATE), 6) AS END_OF_WEEK
	-- Quarter
	, quarter(DATE_ID)                           AS QUARTER
	-- Financial year (FY rolls in July)
	, year(DATE_ID) + CASE WHEN month(DATE_ID) >= 7 THEN 1 ELSE 0 END AS FIN_YEAR
	, concat('FY', year(DATE_ID) + CASE WHEN month(DATE_ID) >= 7 THEN 1 ELSE 0 END) as FIN_YEAR_DESC
	-- ISO year-of-week
	, extract(YEAROFWEEK FROM DATE_ID)           AS ISO_YEAR
	-- Display fields
	, upper(date_format(DATE_ID, 'MMM-yyyy'))    AS MONTH_YEAR
	-- Offsets
	, date_diff(DAY,   DATE_ID, current_date())  * -1 AS DAY_OFFSET
	, date_diff(WEEK,  DATE_ID, current_date())  * -1 AS WEEK_OFFSET
	, date_diff(MONTH, DATE_ID, current_date())  * -1 AS MONTH_OFFSET
	, date_diff(YEAR,  DATE_ID, current_date())  * -1 AS YEAR_OFFSET
FROM workspace.reference.dates
WHERE DATE_ID <= date_add(current_date(), -1)   -- historical only (<= yesterday)
